# Import libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

import concurrent.futures

import seaborn as sns

pd.set_option('max_rows', 300)
pd.set_option('max_columns', 300)

from tqdm.notebook import tqdm

import optiver_feateng3 as opt
import gc

In [ ]:
! ls -hlt ../input/optiver-realized-volatility-prediction/book_train.parquet | wc -l

# Custom functions

In [ ]:
def log_return(list_stock_prices):
    return np.log(list_stock_prices).diff()

In [ ]:
def realized_volatility(series_log_return):
    return np.sqrt(np.sum(series_log_return**2))

# Explore data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

print(train_df.shape)
print(test_df.shape)
print(submit_df.shape)

In [ ]:
train_df.head(2)

In [ ]:
print(f"Unique stock ids: {train_df['stock_id'].nunique()}")
print(f"Unique time ids: {train_df['time_id'].nunique()}")

In [ ]:
book_df = pd.read_parquet(f"/kaggle/input/optiver-realized-volatility-prediction/book_train.parquet/stock_id=0")

print(book_df.shape)
display(book_df.head())

In [ ]:
trade_df = pd.read_parquet(f"/kaggle/input/optiver-realized-volatility-prediction/trade_train.parquet/stock_id=0")

print(trade_df.shape)
display(trade_df.head())

In [ ]:
# 50 MA, 10 MA cross
# OB price vs MA

In [ ]:
# Traded price vs OB (at second)
# Traded price vs OB (above mean, below)
# Num levels on OB trade consumed

# Feature Engineering

In [ ]:
model_cols = ['real_vol1',
 'real_vol4',
 'real_vol2',
 'OB_wap2_dollar_change_abs_sum',
 'OB_wap4_dollar_change_abs_sum',
 'real_vol3',
 'OB_wap1_dollar_change_abs_sum',
 'price_spread_sum',
 'OB_wap3_dollar_change_abs_sum',
 'bid_ask_diff_sum',
 'spread_sum',
 'volume_imbalance_min',
 'stock_id',
 'wapbal_sum',
 'bid_ask_diff_wap2_ratio_mean',
 'spread2_max',
 'spread2_sum',
 'bid_ask_diff2_wap2_ratio_mean',
 'bid_ask_diff_wap_ratio_mean',
 'price_spread2_sum',
 'bid_spread_min',
 'wap1_range']

# Order Book features

In [ ]:
stockLst = train_df['stock_id'].unique().tolist()

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(opt.order_book_features, stockLst), total=len(stockLst)))    

In [ ]:
import functools

stockLst = train_df['stock_id'].unique().tolist()

partial_order_book_features_500 = functools.partial(opt.order_book_features, since_second=500)
partial_order_book_features_400 = functools.partial(opt.order_book_features, since_second=400)
partial_order_book_features_300 = functools.partial(opt.order_book_features, since_second=300)
partial_order_book_features_200 = functools.partial(opt.order_book_features, since_second=200)
partial_order_book_features_100 = functools.partial(opt.order_book_features, since_second=100)


In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_500 = list(tqdm(executor.map(partial_order_book_features_500, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df = pd.concat(results)
opt_train_df_500 = pd.concat(results_500)

print(opt_train_df.shape)

del results, results_500
_ = gc.collect()

In [ ]:
opt_train_df_500.columns = [f'{col}_500' if col not in ['time_id','stock_id'] else col for col in opt_train_df_500.columns]

opt_train_df = opt.additional_feats(opt_train_df, suffix = "")
opt_train_df_500 = opt.additional_feats(opt_train_df_500, suffix = "_500")

opt_train_df = pd.merge(opt_train_df, opt_train_df_500, on=["time_id","stock_id"], how="left")

del opt_train_df_500
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_400 = list(tqdm(executor.map(partial_order_book_features_400, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_400 = pd.concat(results_400)

del results_400
_ = gc.collect()

opt_train_df_400.columns = [f'{col}_400' if col not in ['time_id','stock_id'] else col for col in opt_train_df_400.columns]

opt_train_df_400 = opt.additional_feats(opt_train_df_400, suffix = "_400")

opt_train_df = pd.merge(opt_train_df, opt_train_df_400, on=["time_id","stock_id"], how="left")

del opt_train_df_400
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_300 = list(tqdm(executor.map(partial_order_book_features_300, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_300 = pd.concat(results_300)

del results_300
_ = gc.collect()

opt_train_df_300.columns = [f'{col}_300' if col not in ['time_id','stock_id'] else col for col in opt_train_df_300.columns]

opt_train_df_300 = opt.additional_feats(opt_train_df_300, suffix = "_300")

opt_train_df = pd.merge(opt_train_df, opt_train_df_300, on=["time_id","stock_id"], how="left")

del opt_train_df_300
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_200 = list(tqdm(executor.map(partial_order_book_features_200, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_200 = pd.concat(results_200)

del results_200
_ = gc.collect()

opt_train_df_200.columns = [f'{col}_300' if col not in ['time_id','stock_id'] else col for col in opt_train_df_200.columns]

opt_train_df_200 = opt.additional_feats(opt_train_df_200, suffix = "_300")

opt_train_df = pd.merge(opt_train_df, opt_train_df_200, on=["time_id","stock_id"], how="left")

del opt_train_df_200
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_100 = list(tqdm(executor.map(partial_order_book_features_100, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_100 = pd.concat(results_100)

del results_100
_ = gc.collect()

opt_train_df_100.columns = [f'{col}_300' if col not in ['time_id','stock_id'] else col for col in opt_train_df_100.columns]

opt_train_df_100 = opt.additional_feats(opt_train_df_100, suffix = "_300")

opt_train_df = pd.merge(opt_train_df, opt_train_df_100, on=["time_id","stock_id"], how="left")

del opt_train_df_100
_ = gc.collect()

opt_train_df.shape

# Trade book features

# Recreate metrics from starter notebook

https://www.kaggle.com/jiashenliu/introduction-to-financial-concepts-and-data#Naive-prediction:-using-past-realized-volatility-as-target

In [ ]:
opt_train_df = pd.merge(opt_train_df, train_df, on=['stock_id','time_id'], how="left")

opt_train_df.shape

In [ ]:
from sklearn.metrics import r2_score
def rmspe(y_true, y_pred):
    return  (np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))
R2 = round(r2_score(y_true = opt_train_df['target'], y_pred = opt_train_df['real_vol1']),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'], y_pred = opt_train_df['real_vol1']),3)
print(f'Performance of the naive prediction: R2 score: {R2}, RMSPE: {RMSPE}')

# Save file

In [ ]:
opt_train_df.to_feather("optiver_train.feather")

In [ ]:
!ls -hlt /kaggle/working/